# Churn Model Pipeline

Prototype notebook for converting a simple churn-scoring workflow into a repeatable batch pipeline.


In [1]:
SEED = 7
TRAIN_SPLIT = 0.8
CHURN_THRESHOLD = 0.55
PIPELINE_ORDER = ["load_customer_rows", "build_features", "train_model", "score_model"]


In [2]:
def load_customer_rows():
    return [
        {"customer_id": 101, "tenure_months": 2, "tickets_last_30d": 4, "plan": "starter"},
        {"customer_id": 102, "tenure_months": 18, "tickets_last_30d": 1, "plan": "growth"},
        {"customer_id": 103, "tenure_months": 6, "tickets_last_30d": 3, "plan": "starter"},
        {"customer_id": 104, "tenure_months": 24, "tickets_last_30d": 0, "plan": "enterprise"},
    ]


In [3]:
def build_features(rows):
    feature_rows = []
    for row in rows:
        plan_weight = {"starter": 0.25, "growth": 0.1, "enterprise": -0.05}.get(row["plan"], 0.0)
        risk_signal = (row["tickets_last_30d"] * 0.12) + (1 / max(row["tenure_months"], 1)) + plan_weight
        feature_rows.append({
            "customer_id": row["customer_id"],
            "risk_signal": round(risk_signal, 3),
            "plan": row["plan"],
        })
    return feature_rows


In [4]:
def train_model(feature_rows):
    average_signal = sum(row["risk_signal"] for row in feature_rows) / len(feature_rows)
    return {
        "seed": SEED,
        "train_split": TRAIN_SPLIT,
        "average_signal": round(average_signal, 3),
        "threshold": CHURN_THRESHOLD,
    }


In [5]:
def score_model(model, feature_rows):
    at_risk = [row for row in feature_rows if row["risk_signal"] >= model["threshold"]]
    return {
        "at_risk_customers": [row["customer_id"] for row in at_risk],
        "score_summary": {
            "average_signal": model["average_signal"],
            "threshold": model["threshold"],
            "at_risk_count": len(at_risk),
        },
    }
